### Transformations applied to Results Dataset 


The notebook below accomplishes the following:

1. Read bronze `results` table
1. Kept columns required for analytics (Drop `url` column)
1. Standardized column names using snake_case
1. Renamed columns to make them more meaningful 
1. Filtered out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Removed duplicate records
1. Transformed values of column `race_name` to Title Case
1. Wrote the transformed data to silver `results` table

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
# Read & Write tables for Races

bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df=(
    spark.read.table(bronze_table)
                .drop("url")
                .withColumnsRenamed(
                        {'date':'race_date',
                        'raceName':'race_name',
                        'constructorId':'constructor_Id',
                        'driverId':'driver_Id',
                        'grid':'grid_position',
                        'laps':'completed_laps',
                        'number':'car_number',
                        'poisition':'final_position',
                        'positionText':'final_position_text'
                        }

))
display(results_df)

In [0]:
# finding Duplicates

#'''
dupes= (results_df.groupBy(results_df.columns)
.count()
.filter(F.col("count")>1))
display(dupes)
#'''


In [0]:
# Chaining filters and dropping duplicates

results_valid_df= (
    results_df
            .filter(F.col('season').isNotNull() &
                    F.col('round').isNotNull() &
                    F.col('constructor_id').isNotNull() &
                    F.col('driver_id').isNotNull())
            #.dropDuplicates(['season','round','constructor_id','driver_id'])
            .dropDuplicates()
)

In [0]:
#Count of duplicate records

display(results_df.count()-results_valid_df.count())

In [0]:
#Standardizing Column naming convention
results_final_df=(results_valid_df
                                .withColumn("race_name",F.initcap(F.col("race_name")))    
                )

In [0]:
#Writing Final table into Silver Layer

(

results_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(silver_table)
)


In [0]:
display(results_final_df)